# local-rag 라벨링 도구 & 전자동 배치 파이프라인 (Google Colab + 드라이브 폴더 백업)

Colab T4 GPU 환경에서 PDF 구조 추출, EasyOCR(GPU 가속), 그리고 **Qwen2.5-VL 비전 언어 모델을 활용한 자동 설명 생성**까지 실행합니다.

> **필수:** 상단 메뉴 **런타임 → 런타임 유형 변경 → GPU (T4)** (또는 VS Code 우측 상단 커널에서 Colab T4 GPU)가 연결되어 있는지 확인하세요.

## 1. Drive 마운트 + 프로젝트 로컬 복사

Google Drive의 `MyDrive/local-rag` 폴더를 Colab 초고속 로컬 가상머신 디스크(`/content/local-rag`)로 복사합니다.
(SQLite 잠금 및 디스크 I/O 속도 문제 방지)

In [ ]:
from google.colab import drive
from pathlib import Path
import os, shutil

drive.mount("/content/drive")

# Google Drive 내 프로젝트 경로 (필요시 수정)
DRIVE_ROOT = Path("/content/drive/MyDrive/local-rag")
WORK = Path("/content/local-rag")

assert DRIVE_ROOT.is_dir(), f"Drive에 프로젝트가 없습니다: {DRIVE_ROOT}"

if WORK.exists():
    shutil.rmtree(WORK)

# 불필요한 무거운 폴더 제외하고 복사
shutil.copytree(
    DRIVE_ROOT,
    WORK,
    ignore=shutil.ignore_patterns(
        ".venv", "__pycache__", ".git", "models", "llama", "*.gguf", "node_modules"
    ),
)
os.chdir(WORK)
print("작업 디렉터리 준비 완료:", WORK)

## 2. 의존성 설치 (Java + 라벨러 + VLM 서빙 패키지)

In [ ]:
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

# 1) PDF 구조 분석용 OpenDataLoader Java 런타임
!apt-get -qq update && apt-get -qq install -y openjdk-17-jre-headless

# 2) 라벨러 기본 의존성 (pypdfium2, easyocr, opendataloader-pdf 등)
pip("-r", "tools/labeler/requirements.txt")

# 3) Qwen2.5-VL 서빙용 경량 패키지
pip("transformers", "accelerate", "fastapi", "uvicorn")

import torch
print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"사용 GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ 경고: GPU가 감지되지 않았습니다. 런타임 유형을 GPU로 변경하세요.")

## 3. Drive에서 이전 라벨링 데이터 복원 (있으면)

In [ ]:
from pathlib import Path
import shutil

src = DRIVE_ROOT / "data" / "labeling"
dst = WORK / "data" / "labeling"
dst.mkdir(parents=True, exist_ok=True)

if (src / "labeler.sqlite3").is_file():
    shutil.copy2(src / "labeler.sqlite3", dst / "labeler.sqlite3")
    print("기존 라벨링 DB 복원 완료")
else:
    print("새 라벨링 세션 시작 (기존 DB 없음)")

src_img = src / "images"
if src_img.is_dir():
    if (dst / "images").exists():
        shutil.rmtree(dst / "images")
    shutil.copytree(src_img, dst / "images")
    print("크롭 이미지 복원 완료")

docs = list((WORK / "data" / "docs").rglob("*.pdf")) if (WORK / "data" / "docs").exists() else []
print(f"발견된 PDF 문서: {len(docs)}개")

## 4. Qwen2.5-VL 서버 기동 (Colab GPU)

Colab의 T4 GPU에 `Qwen2.5-VL-3B-Instruct` 모델을 로드하여 로컬 OpenAI 호환 API(`http://127.0.0.1:8090/v1`)로 백그라운드 기동합니다.

In [ ]:
import subprocess, sys, time, urllib.request, json

print("Qwen2.5-VL 모델 서버 기동 중... (약 30~60초 소요)")
vlm_proc = subprocess.Popen(
    [sys.executable, "tools/labeler/vlm_server.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# 서버가 준비될 때까지 대기
ready = False
for _ in range(60):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8090/health", timeout=2) as resp:
            if resp.status == 200:
                ready = True
                break
    except Exception:
        pass
    time.sleep(2)

if ready:
    print("✓ Qwen2.5-VL 서버가 정상적으로 활성화되었습니다! (http://127.0.0.1:8090)")
else:
    print("⚠️ 서버 기동 대기 시간 초과. 로그 확인 필요.")

## 5. [추천] 명령어 한 줄로 전체 문서 일괄 처리 (배치 파이프라인)

웹 화면에서 문서를 하나하나 누를 필요 없이, `data/docs` 내의 **모든 PDF 문서를 한 번에 일괄 추출 + OCR + VLM 설명 생성**합니다.

- `--vlm`: Qwen2.5-VL을 사용해 모든 그림/다이어그램/표의 설명까지 자동 생성하여 DB에 채웁니다.
- `--force`: 이미 추출된 문서도 처음부터 다시 추출할 때 붙입니다.

In [ ]:
# 모든 PDF 일괄 추출 + EasyOCR(GPU) + Qwen2.5-VL 설명 생성
!python tools/labeler/batch_process.py --vlm

## 6. (선택) 웹 UI로 검토 및 라벨링 확인

위 배치 작업이 끝난 뒤 또는 직접 눈으로 보면서 적합/부적합을 라벨링하고 싶을 때 실행합니다.
셀을 실행하면 표시되는 **라벨링 UI 링크**를 클릭하세요.

In [ ]:
import threading, time, subprocess, sys
from google.colab.output import eval_js

PORT = 8788
server_proc = None

def run_server():
    global server_proc
    server_proc = subprocess.Popen(
        [sys.executable, "tools/labeler/server.py"],
        cwd=str(WORK),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for line in server_proc.stdout:
        print(line, end="")

t = threading.Thread(target=run_server, daemon=True)
t.start()
time.sleep(3)

url = eval_js(f"google.colab.kernel.proxyPort({PORT})")
print("\n라벨링 UI 접속 주소:", url)

## 7. Google Drive로 작업 결과 백업 (필수)

Colab 세션이 끊기면 가상머신 디스크는 초기화됩니다.
**작업 완료 후 반드시 이 셀을 실행하여 Drive로 저장하세요.**

In [ ]:
from pathlib import Path
import shutil
from datetime import datetime, timezone

src = WORK / "data" / "labeling"
dst = DRIVE_ROOT / "data" / "labeling"
dst.mkdir(parents=True, exist_ok=True)

db = src / "labeler.sqlite3"
if db.is_file():
    shutil.copy2(db, dst / "labeler.sqlite3")
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    shutil.copy2(db, dst / f"labeler.{stamp}.sqlite3.bak")
    print("DB 파일 백업 완료:", dst / "labeler.sqlite3")

img_src = src / "images"
img_dst = dst / "images"
if img_src.is_dir():
    if img_dst.exists():
        shutil.rmtree(img_dst)
    shutil.copytree(img_src, img_dst)
    print("크롭 이미지 백업 완료:", img_dst)

print("\n모든 라벨링 데이터가 Google Drive에 성공적으로 백업되었습니다!")